# Análise de Risco de Inadimplência — Clínica

**Objetivo:** treinar um modelo de classificação que identifique o risco de inadimplência dos pacientes (Alto / Médio / Baixo), a partir do histórico de pagamentos.

**Etapas do projeto (ETL + Análise + ML):**
1. Extração dos dados (.csv exportado do MySQL)
2. Transformação (criação de features)
3. Tratamento (normalização e outliers)
4. Carga (salvar dataset tratado em .csv)
5. Visualizações com Matplotlib
6. Análise estatística com NumPy
7. Classificação com scikit-learn (3 classes)

> **Observação sobre o dataset:** a base fornecida contém 15 registros de pagamento, um por paciente, todos com situação "Pago". Não há repetição de convênio nem múltiplas consultas por paciente. Isso limita a robustez estatística do modelo (poucos dados para treino/teste), mas o pipeline completo foi implementado exatamente como pedido, usando os dados reais disponíveis, sem inventar dados adicionais.

## 1. Extração — Carregar histórico de pagamentos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Extração: leitura do arquivo .csv exportado do banco MySQL
df = pd.read_csv('pagamentos_clinica_rare.csv')

print(f"Total de registros: {len(df)}")
df.head(15)

In [ ]:
# Conferindo tipos de dados e valores nulos
df.info()
print("\nValores nulos por coluna:")
print(df.isnull().sum())

## 2. Transformação — Criação de Features

Vamos criar as features pedidas:
- **atraso_dias**: diferença entre data de pagamento e data de vencimento (em dias)
- **atraso_medio**: como cada paciente tem apenas um pagamento no dataset, o atraso médio por paciente é o próprio atraso daquele pagamento (documentado como limitação)
- **numero_atrasos**: quantidade de pagamentos em atraso (atraso_dias > 0) por paciente
- **tipo_plano**: já existe como `nome_convenio`, será usada como variável categórica

In [ ]:
# Convertendo as colunas de data para datetime
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'])
df['data_pagamento'] = pd.to_datetime(df['data_pagamento'])

# Feature: atraso em dias (pode ser negativo = pagou antes do vencimento)
df['atraso_dias'] = (df['data_pagamento'] - df['data_vencimento']).dt.days

df[['paciente', 'nome_convenio', 'data_vencimento', 'data_pagamento', 'atraso_dias']]

In [ ]:
# Feature: atraso_medio por paciente
# (neste dataset cada paciente tem 1 pagamento, então atraso_medio = atraso_dias daquele registro;
#  o código já está preparado para agregar corretamente caso existam múltiplos pagamentos por paciente no futuro)
atraso_medio = df.groupby('id_paciente')['atraso_dias'].mean().rename('atraso_medio')

# Feature: numero_atrasos (quantidade de pagamentos feitos em atraso, atraso_dias > 0)
numero_atrasos = df.groupby('id_paciente')['atraso_dias'].apply(lambda x: (x > 0).sum()).rename('numero_atrasos')

df = df.merge(atraso_medio, on='id_paciente').merge(numero_atrasos, on='id_paciente')

# Feature: tipo de plano (renomeando para deixar mais claro)
df['tipo_plano'] = df['nome_convenio']

df[['paciente', 'tipo_plano', 'atraso_dias', 'atraso_medio', 'numero_atrasos']]

## 3. Tratamento — Normalização e Outliers

- Valores de atraso negativos (pagamento antecipado) são tratados como "sem atraso" (0) para fins de cálculo de risco, mas mantidos na coluna original para não perder informação.
- Verificação de outliers usando o método do intervalo interquartil (IQR).
- Normalização das variáveis numéricas com `MinMaxScaler` do scikit-learn.

In [ ]:
# Coluna auxiliar: atraso "efetivo" (não negativo) usada nas regras de risco
df['atraso_efetivo'] = df['atraso_dias'].clip(lower=0)

# Detecção de outliers pelo método IQR na coluna atraso_dias
Q1 = df['atraso_dias'].quantile(0.25)
Q3 = df['atraso_dias'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df['atraso_dias'] < limite_inferior) | (df['atraso_dias'] > limite_superior)]
print(f"Limite inferior: {limite_inferior:.2f} | Limite superior: {limite_superior:.2f}")
print(f"Outliers encontrados: {len(outliers)}")
outliers[['paciente', 'atraso_dias']]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalização (0 a 1) das colunas numéricas relevantes
scaler = MinMaxScaler()
colunas_numericas = ['atraso_efetivo', 'numero_atrasos', 'valor']
df_normalizado = df.copy()
df_normalizado[[c + '_norm' for c in colunas_numericas]] = scaler.fit_transform(df[colunas_numericas])

df_normalizado[['paciente'] + colunas_numericas + [c + '_norm' for c in colunas_numericas]]

## 4. Análise NumPy — Média e Desvio Padrão do Atraso

In [ ]:
atrasos = df['atraso_dias'].to_numpy()

media_atraso = np.mean(atrasos)
desvio_atraso = np.std(atrasos)
mediana_atraso = np.median(atrasos)

print(f"Média do atraso (dias): {media_atraso:.2f}")
print(f"Desvio padrão do atraso (dias): {desvio_atraso:.2f}")
print(f"Mediana do atraso (dias): {mediana_atraso:.2f}")
print(f"Atraso mínimo: {atrasos.min()} dias | Atraso máximo: {atrasos.max()} dias")

## 5. Classificação de Risco (regra de negócio → rótulo para treino)

Como o dataset não tem uma coluna de risco pré-existente, criamos os rótulos (target) a partir de uma regra de negócio baseada no atraso, para então treinar o modelo de Machine Learning que aprenda esse padrão:

- **Baixo risco:** atraso ≤ 0 dias (pagou em dia ou antecipado)
- **Médio risco:** atraso entre 1 e 5 dias
- **Alto risco:** atraso > 5 dias

In [ ]:
def classificar_risco(atraso):
    if atraso <= 0:
        return 'Baixo'
    elif atraso <= 5:
        return 'Médio'
    else:
        return 'Alto'

df['risco'] = df['atraso_dias'].apply(classificar_risco)

print(df['risco'].value_counts())
df[['paciente', 'tipo_plano', 'atraso_dias', 'risco']]

## 6. Visualizações (Matplotlib)

### 6.1 Gráfico de barras — Percentual de atrasos por plano de saúde

In [ ]:
# Percentual de pagamentos em atraso (atraso_dias > 0) por plano de saúde
df['em_atraso'] = df['atraso_dias'] > 0

percentual_atraso_plano = (
    df.groupby('tipo_plano')['em_atraso']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
percentual_atraso_plano.plot(kind='bar', color='#4C72B0')
plt.title('Percentual de Atrasos por Plano de Saúde')
plt.xlabel('Plano de Saúde')
plt.ylabel('% de Pagamentos em Atraso')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('grafico_barras_atraso_por_plano.png', dpi=150)
plt.show()

### 6.2 Gráfico de pizza — Distribuição de status de pagamento (risco)

In [ ]:
distribuicao_risco = df['risco'].value_counts()

cores = {'Baixo': '#55A868', 'Médio': '#DD8452', 'Alto': '#C44E52'}
cores_ordenadas = [cores[c] for c in distribuicao_risco.index]

plt.figure(figsize=(7, 7))
plt.pie(
    distribuicao_risco,
    labels=distribuicao_risco.index,
    autopct='%1.1f%%',
    colors=cores_ordenadas,
    startangle=90
)
plt.title('Distribuição de Status de Pagamento (Nível de Risco)')
plt.axis('equal')
plt.tight_layout()
plt.savefig('grafico_pizza_distribuicao_risco.png', dpi=150)
plt.show()

## 7. Classificação com Machine Learning (scikit-learn)

Modelo escolhido: **Random Forest Classifier**, adequado para classificação multiclasse com poucas features.

> ⚠️ **Observação importante sobre o tamanho da amostra:** o dataset possui apenas 15 registros. Isso é insuficiente para uma avaliação estatisticamente confiável de um modelo de Machine Learning (o ideal seria ter centenas de registros). O pipeline abaixo está tecnicamente correto e completo, mas os resultados de acurácia devem ser interpretados com cautela — servem para demonstrar a metodologia, não para uso real em produção.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

# Codificando a variável categórica tipo_plano
le_plano = LabelEncoder()
df['tipo_plano_cod'] = le_plano.fit_transform(df['tipo_plano'])

# Features (X) e alvo (y)
X = df[['atraso_efetivo', 'numero_atrasos', 'tipo_plano_cod', 'valor']]
y = df['risco']

print(X.shape, y.shape)
X.head()

In [ ]:
# Separação treino/teste
# stratify não é usado aqui pois algumas classes têm poucos exemplos neste dataset pequeno
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Treino: {len(X_train)} registros | Teste: {len(X_test)} registros")

In [ ]:
# Treinamento do modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de classificação:\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Matriz de confusão
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_estimator(
    modelo, X_test, y_test,
    labels=['Baixo', 'Médio', 'Alto'],
    cmap='Blues', ax=ax
)
plt.title('Matriz de Confusão - Risco de Inadimplência')
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150)
plt.show()

In [ ]:
# Importância de cada feature para o modelo
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importancias.plot(kind='barh', color='#4C72B0')
plt.title('Importância das Features no Modelo')
plt.xlabel('Importância')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('importancia_features.png', dpi=150)
plt.show()

importancias

## 8. Carga — Salvar dataset tratado em .csv

In [ ]:
colunas_finais = [
    'id_pagamento', 'id_paciente', 'paciente', 'tipo_plano',
    'data_vencimento', 'data_pagamento', 'valor', 'forma_pagamento',
    'situacao', 'atraso_dias', 'atraso_efetivo', 'atraso_medio',
    'numero_atrasos', 'em_atraso', 'risco'
]

df_final = df[colunas_finais]
df_final.to_csv('dataset_risco_inadimplencia_tratado.csv', index=False)

print("Dataset salvo com sucesso: dataset_risco_inadimplencia_tratado.csv")
df_final.head(15)

## 9. Conclusão

- O pipeline de ETL (Extração, Transformação, Tratamento e Carga) foi implementado por completo.
- As features `atraso_medio`, `numero_atrasos` e `tipo_plano` foram criadas conforme solicitado.
- As visualizações de barras (atraso por plano) e pizza (distribuição de risco) foram geradas com Matplotlib.
- A média e o desvio padrão do atraso foram calculados com NumPy.
- Um classificador Random Forest (scikit-learn) foi treinado para prever o risco de inadimplência em 3 classes (Alto, Médio, Baixo).
- **Limitação:** o dataset fornecido tem apenas 15 registros (um por paciente, todos pagos), o que restringe a confiabilidade estatística do modelo. Para uso real, recomenda-se coletar um histórico maior, com múltiplos pagamentos por paciente ao longo do tempo.